In [2]:
# ---------------------------------------------------------------
# Imports and Configuration
# ---------------------------------------------------------------
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)

INPUT_PATH = Path("cleaned_transformed.parquet")  # produced by the cleaning/transformation notebook

In [3]:
# ---------------------------------------------------------------
# Load cleaned/transformed data
# ---------------------------------------------------------------
data = pd.read_parquet(INPUT_PATH)
print(f"Loaded: {data.shape}")

Loaded: (12612960, 83)


In [4]:
# ---------------------------------------------------------------
# Recreate engineered features this notebook depends on
# (n_chillers_on is required for the regime-conditioned MI check
# below; full feature engineering lives in its own section/notebook,
# this is the minimal subset needed here)
# ---------------------------------------------------------------
data["n_chillers_on"] = data[["CHL_STA_1", "CHL_STA_2", "CHL_STA_3"]].sum(axis=1)

In [5]:
# ---------------------------------------------------------------
# Define candidate feature set: numeric columns minus zero-variance
# columns (from cleaning notebook) and label/metadata columns
# ---------------------------------------------------------------
zero_variance_cols = ["CHL_STA_1", "CT_STA_1", "CWL_SEC_DPSPT", "CWL_SEC_PM_STA_1"]
label_and_metadata_cols = [
    "Datetime", "source_file", "fault_type", "fault_intensity",
    "intensity_degC", "intensity_pct",
]

numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
model_feature_cols = [
    c for c in numeric_cols
    if c not in zero_variance_cols and c not in label_and_metadata_cols
]
print(f"Candidate feature set: {len(model_feature_cols)} columns")

Candidate feature set: 74 columns


In [6]:
# ---------------------------------------------------------------
# Correlation-based redundancy screening
# ---------------------------------------------------------------
corr_matrix = data[model_feature_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns for row in upper.index
    if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.95
]
print(f"Highly correlated pairs (|r| > 0.95): {len(high_corr_pairs)}")
for col, row, r in sorted(high_corr_pairs, key=lambda x: -x[2])[:20]:
    print(f"  {row} <-> {col}: r={r:.3f}")

Highly correlated pairs (|r| > 0.95): 160
  CHL_CD_FLOW_3 <-> CHL_CW_FLOW_3: r=1.000
  CT_FAN_SPD_1 <-> CT_FAN_SPD_CTRL_1: r=1.000
  CT_FAN_SPD_2 <-> CT_FAN_SPD_CTRL_2: r=1.000
  CT_FAN_SPD_3 <-> CT_FAN_SPD_CTRL_3: r=1.000
  CHL_STA_2 <-> CT_STA_2: r=1.000
  CHL_STA_3 <-> CT_STA_3: r=1.000
  CDWL_CW_FLOW <-> CWL_PRI_CW_FLOW: r=1.000
  CHL_CD_FLOW_2 <-> CHL_CW_FLOW_2: r=1.000
  CHL_CD_FLOW_1 <-> CHL_CW_FLOW_1: r=1.000
  CDWL_RW_TEMP <-> CT_RW_TEMP_1: r=1.000
  CHL_RW_TEMP_1 <-> CWL_PRI_RW_TEMP: r=1.000
  CDWL_SW_TEMP <-> CHL_RWCD_TEMP_1: r=1.000
  CWL_PRI_CW_FLOW <-> n_chillers_on: r=1.000
  CDWL_CW_FLOW <-> n_chillers_on: r=1.000
  CDWL_RW_TEMP <-> CHL_SWCD_TEMP_1: r=1.000
  CHL_SWCD_TEMP_1 <-> CT_RW_TEMP_1: r=1.000
  CHL_CD_FLOW_2 <-> CHL_STA_2: r=1.000
  CHL_CD_FLOW_2 <-> CT_STA_2: r=1.000
  CHL_CW_FLOW_2 <-> CHL_STA_2: r=1.000
  CHL_CW_FLOW_2 <-> CT_STA_2: r=1.000


In [7]:
# ---------------------------------------------------------------
# Group all highly-correlated pairs into connected components,
# so redundancy decisions are consistent across chains of pairs
# rather than handled independently (which could keep two
# features that are both redundant with a common third feature)
# ---------------------------------------------------------------
import networkx as nx

G = nx.Graph()
G.add_edges_from([(row, col) for col, row, r in high_corr_pairs])

clusters = list(nx.connected_components(G))
print(f"{len(clusters)} redundant feature clusters found from {len(high_corr_pairs)} pairs")
for cluster in sorted(clusters, key=len, reverse=True):
    print(f"\nCluster ({len(cluster)} features): {sorted(cluster)}")

11 redundant feature clusters found from 160 pairs

Cluster (15 features): ['CDWL_CW_FLOW', 'CDWL_PM_POW_2', 'CHL_CD_FLOW_2', 'CHL_COMP_SPD_CTRL_2', 'CHL_CW_FLOW_2', 'CHL_POW_2', 'CHL_STA_2', 'CT_FAN_SPD_2', 'CT_FAN_SPD_CTRL_2', 'CT_FLOW_2', 'CT_POW_2', 'CT_STA_2', 'CWL_PRI_CW_FLOW', 'CWL_PRI_PM_POW_2', 'n_chillers_on']

Cluster (11 features): ['CDWL_PM_POW_3', 'CHL_CD_FLOW_3', 'CHL_COMP_SPD_CTRL_3', 'CHL_CW_FLOW_3', 'CHL_POW_3', 'CHL_STA_3', 'CT_FAN_SPD_3', 'CT_FAN_SPD_CTRL_3', 'CT_FLOW_3', 'CT_STA_3', 'CWL_PRI_PM_POW_3']

Cluster (6 features): ['CWL_SEC_CW_FLOW', 'CWL_SEC_PM_POW_1', 'CWL_SEC_PM_POW_2', 'CWL_SEC_PM_SPD_1', 'CWL_SEC_PM_SPD_2', 'CWL_SEC_PM_STA_2']

Cluster (5 features): ['CDWL_RW_TEMP', 'CDWL_SW_TEMP', 'CHL_RWCD_TEMP_1', 'CHL_SWCD_TEMP_1', 'CT_RW_TEMP_1']

Cluster (3 features): ['CHL_CD_FLOW_1', 'CHL_CW_FLOW_1', 'CWL_PRI_PM_POW_1']

Cluster (3 features): ['CHL_SWCD_TEMP_3', 'CT_RW_TEMP_3', 'CT_SW_TEMP_3']

Cluster (3 features): ['CHL_RW_TEMP_1', 'CWL_PRI_RW_TEMP', 'CWL_

In [11]:
# ---------------------------------------------------------------
# Remove highly redundant features based on correlation screening
# and physical interpretability
# ---------------------------------------------------------------

redundant_features_to_remove = [

    # -----------------------------------------------------------
    # Cooling tower fan control signals
    # Keep actual fan speed; remove corresponding control signal
    # -----------------------------------------------------------
    "CT_FAN_SPD_CTRL_1",
    "CT_FAN_SPD_CTRL_2",
    "CT_FAN_SPD_CTRL_3",

    # -----------------------------------------------------------
    # Cooling tower status variables
    # Plant operating state is represented by other measurements
    # and the engineered n_chillers_on feature where applicable
    # -----------------------------------------------------------
    "CT_STA_2",
    "CT_STA_3",

    # -----------------------------------------------------------
    # Chiller status variables
    # Individual chiller operating state is represented through
    # physical measurements and n_chillers_on
    # -----------------------------------------------------------
    "CHL_STA_2",
    "CHL_STA_3",

    # -----------------------------------------------------------
    # Chiller 2 redundant variables
    # Retain representative physical measurements:
    #   CHL_POW_2
    #   CHL_CW_FLOW_2
    #   CHL_CD_FLOW_2
    #
    # Remove redundant compressor-speed control measurement
    # -----------------------------------------------------------
    "CHL_COMP_SPD_CTRL_2",

    # -----------------------------------------------------------
    # Chiller 3 redundant variables
    # Retain representative physical measurements:
    #   CHL_POW_3
    #   CHL_CW_FLOW_3
    #   CHL_CD_FLOW_3
    #
    # Remove redundant compressor-speed control measurement
    # -----------------------------------------------------------
    "CHL_COMP_SPD_CTRL_3",

    # -----------------------------------------------------------
    # Primary chilled-water flow redundancy
    # Retain the plant-level primary flow measurement
    # -----------------------------------------------------------
    "CDWL_CW_FLOW",

    # -----------------------------------------------------------
    # Secondary chilled-water loop redundancy
    # Retain secondary chilled-water flow as the primary
    # representation of loop flow
    # -----------------------------------------------------------
    "CWL_SEC_PM_POW_1",
    "CWL_SEC_PM_POW_2",
    "CWL_SEC_PM_SPD_1",
    "CWL_SEC_PM_SPD_2",
    "CWL_SEC_PM_STA_2",

    # -----------------------------------------------------------
    # Condenser-water / chilled-water temperature redundancy
    # Retain representative temperature measurements
    # -----------------------------------------------------------
    "CDWL_RW_TEMP",
    "CDWL_SW_TEMP",

    # -----------------------------------------------------------
    # Additional primary chilled-water temperature redundancy
    # -----------------------------------------------------------
    "CHL_RW_TEMP_1",
    "CWL_PRI_RW_TEMP",

    "CHL_SW_TEMP_1",
    "CWL_PRI_SW_TEMP",

    # -----------------------------------------------------------
    # Cooling-tower temperature redundancy
    # Retain representative cooling-tower temperatures
    # -----------------------------------------------------------
    "CHL_SWCD_TEMP_3",
    "CT_RW_TEMP_3",

    # -----------------------------------------------------------
    # Highly correlated flow/power variable
    # Retain CT_FLOW_1 as the representative operating measurement
    # -----------------------------------------------------------
    "CDWL_PM_POW_1",
]


# ---------------------------------------------------------------
# Zero-variance columns already removed during cleaning
# ---------------------------------------------------------------

zero_variance_cols = [
    "CHL_STA_1",
    "CT_STA_1",
    "CWL_SEC_DPSPT",
    "CWL_SEC_PM_STA_1"
]


# ---------------------------------------------------------------
# Combine zero-variance and redundant features
# ---------------------------------------------------------------

features_to_remove = (
    redundant_features_to_remove
    + zero_variance_cols
)

# Remove duplicates while preserving order
features_to_remove = list(dict.fromkeys(features_to_remove))


# ---------------------------------------------------------------
# Keep only columns that actually exist in the dataset
# ---------------------------------------------------------------

features_to_remove = [
    col for col in features_to_remove
    if col in data.columns
]


# ---------------------------------------------------------------
# Remove selected features from dataset
# ---------------------------------------------------------------

data_reduced = data.drop(
    columns=features_to_remove
)


# ---------------------------------------------------------------
# Update modeling feature list
# ---------------------------------------------------------------

model_feature_cols_reduced = [
    col for col in model_feature_cols
    if col not in features_to_remove
]


# ---------------------------------------------------------------
# Report results
# ---------------------------------------------------------------

print(f"Original modeling features: {len(model_feature_cols)}")
print(
    f"Removed features: "
    f"{len(features_to_remove)}"
)
print(
    f"Remaining modeling features: "
    f"{len(model_feature_cols_reduced)}"
)

print("\nRemoved features:")
for col in features_to_remove:
    print(f"  - {col}")

Original modeling features: 74
Removed features: 28
Remaining modeling features: 50

Removed features:
  - CT_FAN_SPD_CTRL_1
  - CT_FAN_SPD_CTRL_2
  - CT_FAN_SPD_CTRL_3
  - CT_STA_2
  - CT_STA_3
  - CHL_STA_2
  - CHL_STA_3
  - CHL_COMP_SPD_CTRL_2
  - CHL_COMP_SPD_CTRL_3
  - CDWL_CW_FLOW
  - CWL_SEC_PM_POW_1
  - CWL_SEC_PM_POW_2
  - CWL_SEC_PM_SPD_1
  - CWL_SEC_PM_SPD_2
  - CWL_SEC_PM_STA_2
  - CDWL_RW_TEMP
  - CDWL_SW_TEMP
  - CHL_RW_TEMP_1
  - CWL_PRI_RW_TEMP
  - CHL_SW_TEMP_1
  - CWL_PRI_SW_TEMP
  - CHL_SWCD_TEMP_3
  - CT_RW_TEMP_3
  - CDWL_PM_POW_1
  - CHL_STA_1
  - CT_STA_1
  - CWL_SEC_DPSPT
  - CWL_SEC_PM_STA_1


In [13]:
# ---------------------------------------------------------------
# Finalize feature selection decisions and save for modeling
# ---------------------------------------------------------------

OUTPUT_PATH = Path("feature_selected.parquet")

# Save feature-selected dataset
data_reduced.to_parquet(
    OUTPUT_PATH,
    index=False
)

# Save final modeling feature list
import json

with open("final_feature_cols.json", "w") as f:
    json.dump(
        model_feature_cols_reduced,
        f,
        indent=2
    )

print(f"Saved feature-selected dataset: {OUTPUT_PATH}")
print(
    f"Dataset shape: "
    f"{data_reduced.shape}"
)
print(
    f"Final modeling features: "
    f"{len(model_feature_cols_reduced)}"
)
print("Saved feature list: final_feature_cols.json")

Saved feature-selected dataset: feature_selected.parquet
Dataset shape: (12612960, 56)
Final modeling features: 50
Saved feature list: final_feature_cols.json
